# Maick Dane Nkou — `raw-sw75-ha425-yo55-am55-fr128125`

**Reproduction notebook for the submitted tokenizer.** It trains the tokenizer from
scratch on the official train split of
`Similoluwa/african-multilingual-tokenizer-challenge` @ `v1.0.0` and asserts that
the resulting bytes match the submitted `tokenizer.json` before evaluating it on
the full official validation split.

Recipe:

```text
BPE, vocab 10,000, min_frequency 5
pre-tokenizer: ByteLevel(add_prefix_space=False, use_regex=False)  # raw pieces,
               no whitespace split, so merges may cross word boundaries
decoder: ByteLevel; no normalizer, post-processor, special or UNK tokens
initial alphabet: sorted(ByteLevel.alphabet())  # full 256 byte symbols
weights (en, fr, ha, sw, yo, am): x1, x1.28125, x4.25, x7.5, x5.5, x5.5
            -> units 32, 41, 136, 240, 176, 176 over denominator 32,
               deterministic cumulative schedule (every train row kept)
train rows: 240,000 original -> 1,001,250 weighted (round-robin serialization)
tokenizers==0.22.1
```

Expected result on the 24,000 official validation rows:
**score 1.9204967724666027**, zero UNK, 100% exact reconstruction, zero
guardrail and reconstruction penalties.

Designed for a free CPU runtime (Google Colab). Three independent runs of this
recipe produced identical bytes.

In [ ]:
# Pin the environment before anything else.
!pip install --quiet "tokenizers==0.22.1" "datasets>=4.0,<5"

import csv
import gc
import hashlib
import json
import os
import statistics
import subprocess
import sys
import tempfile
import time
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import tokenizers
from tokenizers import Tokenizer
assert tokenizers.__version__ == "0.22.1", tokenizers.__version__
print("tokenizers", tokenizers.__version__)

In [ ]:
# Frozen configuration. Do not edit when reproducing the submission.
RUN_LABEL = "raw-sw75-ha425-yo55-am55-fr128125"

DATASET_ID = "Similoluwa/african-multilingual-tokenizer-challenge"
DATASET_REVISION = "v1.0.0"
LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
CONTEXT_LANGUAGES = ("en", "fr")
CONTEXT_FERTILITY_RATIO = 1.15
RECONSTRUCTION_PENALTY = 3.0
UNKNOWN_PENALTY = 100.0
CPU_THREADS = 2

# Language weights as integer units over DENOMINATOR, order en, fr, ha, sw, yo, am:
# x1, x1.28125, x4.25, x7.5, x5.5, x5.5
CONFIG = {
    "name": RUN_LABEL,
    "units": {"en": 32, "fr": 41, "ha": 136, "sw": 240, "yo": 176, "am": 176},
    "denominator": 32,
    "vocab_size": 10_000,
    "min_frequency": 5,
}
WEIGHTED_ROWS = sum(40_000 * CONFIG["units"][lang] // CONFIG["denominator"] for lang in LANGUAGES)

# Fingerprints of the expected artifacts (recorded from the submission run).
EXPECTED_TOKENIZER_SHA256 = "4043a0499c1e4578bbc636a91f580abfe77dbe0ac52af4dbb0335743e973f5ed"
EXPECTED_TRAIN_JSONL_SHA256 = "85140695f33ff87ead838dbac449fbc7f13aec72dde4129556d7538ba17cbc4f"
EXPECTED_SCORE = 1.9204967724666027

BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
WORK_DIR = BASE / f"airf-{RUN_LABEL}-runtime"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("work dir:", WORK_DIR)
print("weighted rows:", f"{WEIGHTED_ROWS:,}")

## Official data

The train split is serialised to JSONL in round-robin `en/fr/ha/sw/yo/am` order,
keeping each language's internal order; the validation split keeps source order.
By default both splits are loaded from the Hub at the pinned revision.

**Offline fallback:** if the two official CSV exports are already on disk, set
`LOCAL_DATA_DIR` below to their directory; the notebook then uses them only
after verifying their SHA-256 fingerprints, so the offline path cannot silently
train on different data.

In [ ]:
# Optional offline fallback: directory containing the official train.csv /
# validation.csv exports. Leave as "" to download from the Hub.
LOCAL_DATA_DIR = ""

TRAIN_CSV_SHA256 = "bf807cb78e58a6dfb3ba50652013d54fb1d2f9f5e6eec4c98f22d5989364efd7"
VALIDATION_CSV_SHA256 = "d010df416334c0f9086d0831e7aa19c2a7d233b08032e2a7d3118f512d3c0f00"


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def iter_rows_from_hub(split):
    from datasets import load_dataset
    dataset = load_dataset(DATASET_ID, split=split, revision=DATASET_REVISION)
    for row in dataset:
        yield row["language"], row["text"]


def iter_rows_from_csv(path, expected_sha256):
    actual = sha256_file(path)
    if actual != expected_sha256:
        raise ValueError(f"{path}: sha256 {actual} != expected {expected_sha256}")
    with Path(path).open(encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ["language", "text"]:
            raise ValueError(f"{path}: unexpected columns {reader.fieldnames}")
        for row in reader:
            yield row["language"], row["text"]


def iter_split(split):
    if LOCAL_DATA_DIR:
        expected = TRAIN_CSV_SHA256 if split == "train" else VALIDATION_CSV_SHA256
        yield from iter_rows_from_csv(Path(LOCAL_DATA_DIR) / f"{split}.csv", expected)
    else:
        yield from iter_rows_from_hub(split)

In [ ]:
def dump_pair(stream, language, text):
    stream.write(json.dumps([language, text], ensure_ascii=False, separators=(",", ":")) + "\n")


def prepare_split(split):
    expected_per_language = 40_000 if split == "train" else 4_000
    counts = Counter()
    destination = WORK_DIR / f"{split}.jsonl"
    temporary = destination.with_name(destination.name + ".partial")
    with tempfile.TemporaryDirectory(dir=WORK_DIR) as temporary_dir:
        temporary_dir = Path(temporary_dir)
        language_paths = {lang: temporary_dir / f"{lang}.jsonl" for lang in LANGUAGES}
        streams = {}
        try:
            if split == "train":
                streams = {lang: path.open("w", encoding="utf-8", newline="\n")
                           for lang, path in language_paths.items()}
            with temporary.open("w", encoding="utf-8", newline="\n") as output:
                for language, text in iter_split(split):
                    if language not in LANGUAGES or not isinstance(text, str) or not text.split():
                        raise ValueError(f"Invalid row in {split}")
                    counts[language] += 1
                    dump_pair(streams[language] if split == "train" else output, language, text)
                expected = dict.fromkeys(LANGUAGES, expected_per_language)
                if counts != expected:
                    raise ValueError(f"Unexpected counts for {split}: {dict(counts)}")
                if split == "train":
                    for stream in streams.values():
                        stream.close()
                    readers = [path.open(encoding="utf-8") for path in language_paths.values()]
                    try:
                        for group in zip(*readers, strict=True):
                            output.writelines(group)
                    finally:
                        for reader in readers:
                            reader.close()
        finally:
            for stream in streams.values():
                stream.close()
    os.replace(temporary, destination)
    print(split, {"rows": sum(counts.values()), "sha256": sha256_file(destination)})
    return destination


TRAIN_PATH = prepare_split("train")
VALIDATION_PATH = prepare_split("validation")
if sha256_file(TRAIN_PATH) != EXPECTED_TRAIN_JSONL_SHA256:
    raise ValueError("train serialization does not match the recorded fingerprint")
print("train serialization matches the recorded fingerprint")

## Training worker

The worker is written to a file and executed in a separate process so training
memory is released afterwards. It receives only the train serialization and the
config; it never sees the validation split.

In [ ]:
WORKER_SOURCE = r"""import argparse
import hashlib
import json
import os
import time
from collections import Counter
from pathlib import Path

from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
TOKENIZERS_VERSION = "0.22.1"


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def read_rows(path):
    with Path(path).open(encoding="utf-8") as stream:
        for line in stream:
            language, text = json.loads(line)
            if language not in LANGUAGES or not isinstance(text, str) or not text.split():
                raise ValueError("Invalid train row")
            yield language, text


def weighted_texts(rows, units, denominator):
    if set(units) != set(LANGUAGES):
        raise ValueError("Missing language weight")
    if any(type(value) is not int or value < denominator for value in units.values()):
        raise ValueError("Invalid cumulative weight")
    seen = Counter()
    for language, text in rows:
        index = seen[language]
        seen[language] += 1
        copies = ((index + 1) * units[language]) // denominator - (index * units[language]) // denominator
        if copies < 1:
            raise ValueError("A train row was omitted")
        for _ in range(copies):
            yield text


def train_tokenizer(texts, *, vocab_size, min_frequency, length):
    tokenizer = Tokenizer(models.BPE())
    # Raw ByteLevel pieces: no whitespace split, so merges may cross words.
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(
        add_prefix_space=False, use_regex=False
    )
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_frequency,
        special_tokens=[],
        initial_alphabet=sorted(pre_tokenizers.ByteLevel.alphabet()),
        show_progress=False,
    )
    tokenizer.train_from_iterator(texts, trainer=trainer, length=length)
    return tokenizer


ROUNDTRIP_CASES = [
    "", " ", "   ", "\t\n\r\n", "  Hello  WORLD!\tNext\nline.  ",
    "[UNK] [CLS] [SEP] <0xFF>", "é e\u0301 Ì I\u0300", "👩🏿\u200d💻 🌍 中文 العربية",
    "a\u00a0b\u2003c\u200bd", "\x00\x01\x7f\ufeff\U0010ffff", "don't l’amour — … ።",
]


def strict_failure_count(tokenizer, texts, batch_size=512):
    failures = 0
    texts = list(texts)
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encodings = tokenizer.encode_batch(batch, add_special_tokens=False)
        ids = [encoding.ids for encoding in encodings]
        decoded = tokenizer.decode_batch(ids, skip_special_tokens=False)
        skipped = tokenizer.decode_batch(ids, skip_special_tokens=True)
        failures += sum(
            original != restored or original != restored_skipped
            for original, restored, restored_skipped in zip(batch, decoded, skipped, strict=True)
        )
    return failures


def train_job(train_path, config, output):
    if config["vocab_size"] != 10_000:
        raise ValueError("Unexpected vocabulary size")
    denominator = config["denominator"]
    counts = Counter(language for language, _ in read_rows(train_path))
    if counts != dict.fromkeys(LANGUAGES, 40_000):
        raise ValueError(f"Full official train required: {dict(counts)}")
    weighted_rows = sum(40_000 * config["units"][language] // denominator for language in LANGUAGES)
    before = sha256_file(train_path)
    started = time.perf_counter()
    tokenizer = train_tokenizer(
        weighted_texts(read_rows(train_path), config["units"], denominator),
        vocab_size=config["vocab_size"],
        min_frequency=config["min_frequency"],
        length=weighted_rows,
    )
    if tokenizer.get_vocab_size(with_added_tokens=True) != config["vocab_size"]:
        raise ValueError("Expected exactly 10,000 vocabulary entries")
    temporary = Path(output).with_name(Path(output).name + ".partial")
    tokenizer.save(str(temporary), pretty=True)
    del tokenizer
    reloaded = Tokenizer.from_file(str(temporary))
    if strict_failure_count(reloaded, ROUNDTRIP_CASES):
        raise ValueError("Lossy pipeline on strict regression cases")
    if Path(temporary).stat().st_size > 20 * 1024 * 1024:
        raise ValueError("Tokenizer exceeds 20 MiB")
    if sha256_file(train_path) != before:
        raise ValueError("Train corpus changed during training")
    os.replace(temporary, output)
    atomic_json(Path(output).parent / "training.json", {
        "config": config,
        "train_sha256": before,
        "tokenizer_sha256": sha256_file(output),
        "training_seconds": time.perf_counter() - started,
        "weighted_rows": weighted_rows,
    })


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train", required=True)
    parser.add_argument("--config", required=True, help="JSON config string")
    parser.add_argument("--output", required=True)
    args = parser.parse_args()
    from tokenizers import __version__
    if __version__ != TOKENIZERS_VERSION:
        raise RuntimeError("tokenizers==0.22.1 required")
    train_job(args.train, json.loads(args.config), args.output)


if __name__ == "__main__":
    main()
"""
WORKER_PATH = WORK_DIR / "cpu_train_worker.py"
WORKER_PATH.write_text(WORKER_SOURCE, encoding="utf-8")
print("worker sha256:", sha256_file(WORKER_PATH))

In [ ]:
MODEL_DIR = WORK_DIR / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_PATH = MODEL_DIR / "tokenizer.json"

started = time.perf_counter()
proc = subprocess.run(
    [
        sys.executable, str(WORKER_PATH),
        "--train", str(TRAIN_PATH),
        "--config", json.dumps(CONFIG),
        "--output", str(TOKENIZER_PATH),
    ],
    env={**os.environ, "RAYON_NUM_THREADS": str(CPU_THREADS),
         "TOKENIZERS_PARALLELISM": "true"},
    capture_output=True, text=True, timeout=7200,
)
if proc.returncode != 0:
    raise RuntimeError(proc.stderr[-4000:])
print("training wall time: %.1f s" % (time.perf_counter() - started))
receipt = json.loads((MODEL_DIR / "training.json").read_text(encoding="utf-8"))
print(json.dumps(receipt, indent=2))

In [ ]:
# Byte-level identity gate: the trained file must match the submission.
actual_sha256 = sha256_file(TOKENIZER_PATH)
print(actual_sha256)
assert actual_sha256 == EXPECTED_TOKENIZER_SHA256, (
    "trained tokenizer does not match the submitted bytes; "
    "check dataset revision, tokenizers version and config"
)
print("TRAINED BYTES MATCH THE SUBMITTED tokenizer.json")

## Full official-validation evaluation

Implements exactly the competition metric: tokens per whitespace-separated word
plus 100x UNK rate, averaged over ha/sw/yo/am, plus the English/French
guardrail excess (1.15x the scored mean) and 3x the share of rows not
reconstructed exactly. A stricter exact-match audit runs on every row.

In [ ]:
import statistics as _statistics

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))


def load_validation():
    examples = []
    with VALIDATION_PATH.open(encoding="utf-8") as stream:
        for line in stream:
            language, text = json.loads(line)
            examples.append((language, text))
    counts = Counter(language for language, _ in examples)
    assert counts == dict.fromkeys(LANGUAGES, 4_000), dict(counts)
    return examples


VALIDATION = load_validation()
texts = [text for _, text in VALIDATION]
encodings = tokenizer.encode_batch(texts, add_special_tokens=False)

token_counts = defaultdict(int)
word_counts = defaultdict(int)
lossy = 0
for (language, text), encoding in zip(VALIDATION, encodings, strict=True):
    token_counts[language] += len(encoding.ids)
    word_counts[language] += len(text.split())
    restored = tokenizer.decode(encoding.ids, skip_special_tokens=True)
    if (unicodedata.normalize("NFC", restored).strip()
            != unicodedata.normalize("NFC", text).strip()):
        lossy += 1

fertility = {lang: token_counts[lang] / word_counts[lang] for lang in LANGUAGES}
scored_mean = sum(fertility[lang] for lang in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)
budget = CONTEXT_FERTILITY_RATIO * scored_mean
overages = {lang: max(0.0, fertility[lang] - budget) for lang in CONTEXT_LANGUAGES}
reconstruction = 1 - lossy / len(VALIDATION)
score = scored_mean + sum(overages.values()) + RECONSTRUCTION_PENALTY * (1 - reconstruction)

print(f"{'language':<10}{'tokens':>10}{'words':>10}{'tokens/word':>14}")
for lang in LANGUAGES:
    marker = "*" if lang in SCORED_LANGUAGES else " "
    print(f"{lang + marker:<10}{token_counts[lang]:>10,}{word_counts[lang]:>10,}{fertility[lang]:>14.6f}")
print()
print("scored mean        :", scored_mean)
print("guardrail budget   :", budget)
print("guardrail overages :", overages)
print("reconstruction     :", reconstruction)
print("SCORE              :", score)

In [ ]:
# Stricter audit: every row must decode EXACTLY to itself, both decode modes.
failures = 0
for start in range(0, len(texts), 1024):
    batch = texts[start:start + 1024]
    ids = [e.ids for e in tokenizer.encode_batch(batch, add_special_tokens=False)]
    dec_false = tokenizer.decode_batch(ids, skip_special_tokens=False)
    dec_true = tokenizer.decode_batch(ids, skip_special_tokens=True)
    failures += sum((t != a) or (t != b) for t, a, b in zip(batch, dec_false, dec_true, strict=True))
print("strict exact-match failures:", failures)
assert failures == 0

In [ ]:
# Runtime (informational): median of three full-validation encode passes.
tokenizer.encode_batch(texts, add_special_tokens=False)  # warm-up
timings = []
for _ in range(3):
    started = time.perf_counter()
    tokenizer.encode_batch(texts, add_special_tokens=False)
    timings.append(time.perf_counter() - started)
elapsed = _statistics.median(timings)
total_chars = sum(len(text) for text in texts)
print(f"encode 24,000 rows: {elapsed:.3f}s  ({total_chars / elapsed / 1e6:.2f}M chars/s)")
print("The official 5x character-level-baseline limit is enforced by the")
print("repository tooling; this tokenizer encodes at roughly baseline speed.")

In [ ]:
# Final gates before treating this run as a reproduction of the submission.
assert tokenizer.get_vocab_size(with_added_tokens=True) == 10_000
assert abs(score - EXPECTED_SCORE) < 5e-9, score
assert reconstruction == 1.0
assert all(value == 0.0 for value in overages.values())
print("ALL GATES PASSED")
print("tokenizer:", TOKENIZER_PATH)
print("sha256   :", EXPECTED_TOKENIZER_SHA256)
print("score    :", score)